# Auditing a model on `compas`

A full isitfair audit of a logistic regression fitted to the public
`compas` dataset: discrimination, calibration, decision curves,
Hosmer-Lemeshow, estimability, a recalibration ladder, mitigation and an
HTML report.

Before running, build the prediction table once from the repository root:

    python -I extra_examples/datasets.py compas

`B = 50` bootstrap replicates keep the run short; use 2,000 for results you
intend to report.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "extra_examples" / "datasets.py").exists():
        HERE = _c / "extra_examples"
        break
sys.path.insert(0, str(HERE))
import datasets  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing  # noqa: E402

NAME = "compas"
B = 50
SEED = 42
META = datasets.meta(NAME)
THRESHOLD = META["threshold"]
ATTRS = datasets.ATTRS[NAME]
print(f"isitfair {isitfair.__version__}")
print(f"{NAME}: outcome {META['outcome']}; threshold {THRESHOLD}; B = {B}")

isitfair 0.4.0
compas: outcome two_year_recid == 1 (the COMPAS score itself is not a feature); threshold 0.5; B = 50


## 1. Load the predictions

One row per test-set patient or person: the outcome, the model's predicted
probability, and the attributes to audit.

In [2]:
df = datasets.load(NAME)
print(f"n = {len(df)}, prevalence = {df['y_true'].mean():.3f}")
for a in ATTRS:
    print(f"{a}: {df[a].nunique()} levels, {int(df[a].isna().sum())} missing")

n = 1852, prevalence = 0.455
race: 6 levels, 0 missing
sex: 2 levels, 0 missing
age_cat: 3 levels, 0 missing


isitfair refuses missing subgroup labels rather than silently dropping those
rows. If any attribute has missing labels, this cell shows the error and then
recodes them to an explicit `"Unknown"` level.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    try:
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    except ValueError as exc:
        print("ValueError:", exc)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


## 2. Split off a calibration half for mitigation

The test half is split 50/50, stratified on the outcome. Mitigation
calibrators are fitted on one half and every audit runs on the other, so no
row is scored by a calibrator fitted on it.

In [4]:
i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                 stratify=df["y_true"])
cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
# mitigate() rejects audited levels that the calibration half never saw; drop
# such rows (for example diabetes130's single "Unknown/Invalid" gender).
unseen = np.zeros(len(eval_df), bool)
for a in ATTRS:
    unseen |= ~eval_df[a].isin(set(cal_df[a]))
if unseen.any():
    print(f"dropping {int(unseen.sum())} audited row(s) with levels absent from the calibration half")
    eval_df = eval_df[~unseen].reset_index(drop=True)
print(f"calibration n = {len(cal_df)}, audit n = {len(eval_df)}")

calibration n = 926, audit n = 926


## 3. Construct the audit

Each attribute is audited on its own, and the first two are also crossed
(passing `None` for a tuple key asks isitfair to build the intersection).

In [5]:
y = eval_df["y_true"].to_numpy(dtype=float)
p = eval_df["y_proba"].to_numpy(dtype=float)
marginals = {a: eval_df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None

audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                      n_bootstrap=B, random_state=SEED)
audit

FairnessAudit(n=926, events=421, threshold=0.5, n_bootstrap=50, random_state=42, attributes: race (5), sex (2), age_cat (3), race × sex (9))

## 4. Discrimination

In [6]:
disc = audit.discrimination()
disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                              "auroc_ci_low", "auroc_ci_high"]]

,attribute,subgroup,n,n_events,auroc,auroc_ci_low,auroc_ci_high
0,race,Overall,926.0,421.0,0.709812,0.681999,0.740784
1,race,African-American,477.0,254.0,0.713896,0.676986,0.763150
2,race,Asian,4.0,2.0,1.000000,NaN,NaN
3,race,Caucasian,328.0,121.0,0.655148,0.588672,0.708656
4,race,Hispanic,61.0,29.0,0.759159,0.619301,0.859488
5,race,Other,56.0,15.0,0.620325,NaN,NaN
10,sex,Overall,926.0,421.0,0.709812,0.681999,0.740784
11,sex,Female,185.0,66.0,0.714922,0.611858,0.801587
12,sex,Male,741.0,355.0,0.700646,0.666249,0.737913
17,age_cat,Overall,926.0,421.0,0.709812,0.681999,0.740784


## 5. Calibration

In [7]:
cal = audit.calibration()
cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                            "calibration_slope", "brier", "ece", "ici"]]

,attribute,subgroup,n,calibration_intercept,calibration_slope,brier,ece,ici
0,race,Overall,926.0,-0.001243,0.960563,0.214168,0.023138,0.012004
1,race,African-American,477.0,0.085034,0.925654,0.216957,0.061299,0.029173
2,race,Asian,4.0,0.445948,24.417333,0.147097,0.369502,0.369502
3,race,Caucasian,328.0,-0.080470,0.778603,0.216235,0.079403,0.040571
4,race,Hispanic,61.0,0.275292,1.549092,0.201396,0.121350,0.070664
5,race,Other,56.0,-0.659676,1.100849,0.197010,0.191399,0.142866
8,sex,Overall,926.0,-0.001243,0.960563,0.214168,0.023138,0.012004
9,sex,Female,185.0,-0.200215,1.416611,0.195951,0.133495,0.083369
10,sex,Male,741.0,0.048141,0.877370,0.218716,0.045942,0.016930
13,age_cat,Overall,926.0,-0.001243,0.960563,0.214168,0.023138,0.012004


In [8]:
curves = audit.calibration_curves()
roc = audit.roc_curves()
print("calibration curves:", curves.shape, "| ROC curves:", roc.shape)

calibration curves: (2300, 7) | ROC curves: (2300, 7)


## 6. Decision curve: gaps in net benefit between subgroups

In [9]:
dc = audit.decision_curve()
dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]]

,attribute,gap_value,gap_ci_low,gap_ci_high
120,race,0.577381,NaN,NaN
181,sex,0.162758,0.071200,0.264322
262,age_cat,0.289170,0.198084,0.408005
463,race × sex,0.639423,NaN,NaN


## 7. Estimability

Cells with too few events for a metric are flagged rather than reported.

In [10]:
est = audit.estimability()
print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

87 of 281 cell-metrics not reportable


## 8. Hosmer-Lemeshow

In [11]:
hl = audit.hosmer_lemeshow()
hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                          "hl_p_value", "n_bins_used"]]

,attribute,subgroup,n,hl_statistic,hl_df,hl_p_value,n_bins_used
0,race,Overall,926.0,3.314066,10.0,0.973038,10.0
1,race,African-American,477.0,14.994535,10.0,0.132261,10.0
2,race,Asian,4.0,2.517244,4.0,0.641551,4.0
3,race,Caucasian,328.0,13.053541,10.0,0.220694,10.0
4,race,Hispanic,61.0,6.607626,10.0,0.761895,10.0
5,race,Other,56.0,11.213726,10.0,0.341111,10.0
7,sex,Overall,926.0,3.314066,10.0,0.973038,10.0
8,sex,Female,185.0,22.854466,10.0,0.011297,10.0
9,sex,Male,741.0,11.138108,10.0,0.346860,10.0
11,age_cat,Overall,926.0,3.314066,10.0,0.973038,10.0


## 9. Intersection cells

In [12]:
audit.cell_diagnostics() if INTERSECTION else "single attribute; no intersection"

,attribute,subgroup,n,n_events,shrunk,marginal_rate,cell_rate,shrinkage_factor
0,race × sex,African-American / Female,92,36,True,0.532495,0.391304,0.615626
1,race × sex,African-American / Male,385,218,False,0.532495,0.566234,0.276792
2,race × sex,Asian / Male,4,2,True,0.500000,0.500000,0.973571
3,race × sex,Caucasian / Female,72,25,True,0.368902,0.347222,0.671757
4,race × sex,Caucasian / Male,256,96,False,0.368902,0.375000,0.365315
5,race × sex,Hispanic / Female,8,4,True,0.475410,0.500000,0.948503
6,race × sex,Hispanic / Male,53,25,True,0.475410,0.471698,0.735463
7,race × sex,Other / Female,13,1,True,0.267857,0.076923,0.918927
8,race × sex,Other / Male,43,14,True,0.267857,0.325581,0.774100


## 10. Recalibration ladder on the attribute with the largest intercept gap

A level with no events at all cannot be recalibrated, and isitfair says so.
In that case the ladder moves on to the next attribute.

In [13]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]
ladder = None
for by in RANKED:
    try:
        ladder = audit.recalibration_ladder(by=by)
        print("ladder by", by)
        break
    except ValueError as exc:
        print(f"{by}: {exc}")
ladder

ladder by race


,rung,attribute,subgroup,n,n_events,calibration_intercept,intercept_ci_low,intercept_ci_high,calibration_slope,slope_ci_low,...,brier_ci_low,brier_ci_high,ece,ici,net_benefit,gap_metric,gap_value,event_floor,below_floor,not_estimable_final
0,none,race,Overall,926.0,421.0,-0.001243,-0.129153,0.102420,0.960563,0.736739,...,0.206404,0.226657,0.023138,0.012004,0.126350,NaN,NaN,25.0,False,False
1,none,race,African-American,477.0,254.0,0.085034,-0.054958,0.256209,0.925654,0.643220,...,0.202159,0.233398,0.061299,0.029173,0.190776,NaN,NaN,25.0,False,False
2,none,race,Asian,4.0,2.0,0.445948,NaN,NaN,24.417333,NaN,...,NaN,NaN,0.369502,0.369502,0.500000,NaN,NaN,25.0,True,True
3,none,race,Caucasian,328.0,121.0,-0.080470,-0.268756,0.173480,0.778603,0.522902,...,0.201469,0.232796,0.079403,0.040571,0.045732,NaN,NaN,25.0,False,False
4,none,race,Hispanic,61.0,29.0,0.275292,-0.183049,0.787091,1.549092,NaN,...,0.167965,0.232671,0.121350,0.070664,0.114754,NaN,NaN,25.0,False,True
5,none,race,Other,56.0,15.0,-0.659676,NaN,NaN,1.100849,NaN,...,NaN,NaN,0.191399,0.142866,0.035714,NaN,NaN,25.0,True,True
6,none,race,GAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,max_intercept_gap,1.105624,NaN,NaN,NaN
7,common,race,Overall,926.0,421.0,0.000203,-0.124223,0.105429,0.597080,0.154554,...,0.205389,0.228216,0.033345,0.012122,0.119870,NaN,NaN,25.0,False,False
8,common,race,African-American,477.0,254.0,0.070432,-0.093885,0.249146,0.575232,0.137344,...,0.201018,0.232614,0.093402,0.019901,0.186583,NaN,NaN,25.0,False,False
9,common,race,Asian,4.0,2.0,0.263400,NaN,NaN,29.615341,NaN,...,NaN,NaN,0.343956,0.343956,0.500000,NaN,NaN,25.0,True,True


## 11. Mitigation

Group-wise isotonic recalibration, then Wasserstein post-processing, both
fitted on the calibration half.

In [14]:
calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba"].to_numpy(float),
              {a: cal_df[a].to_numpy(object) for a in ATTRS})
mitigated_gr = audit.mitigate(method=GroupRecalibration(method="isotonic"), calibration_data=calib_data)
a0 = ATTRS[0]
keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
before, after = audit.calibration(), mitigated_gr[a0].calibration()
print("Before,", a0)
print(before[before.attribute == a0][keep].to_string(index=False))
print("\nAfter group isotonic recalibration,", a0)
print(after[after.attribute == a0][keep].to_string(index=False))

Before, race
        subgroup  calibration_intercept  calibration_slope    brier      ece
         Overall              -0.001243           0.960563 0.214168 0.023138
African-American               0.085034           0.925654 0.216957 0.061299
           Asian               0.445948          24.417333 0.147097 0.369502
       Caucasian              -0.080470           0.778603 0.216235 0.079403
        Hispanic               0.275292           1.549092 0.201396 0.121350
           Other              -0.659676           1.100849 0.197010 0.191399
             GAP                    NaN                NaN      NaN      NaN
             GAP                    NaN                NaN      NaN      NaN

After group isotonic recalibration, race
        subgroup  calibration_intercept  calibration_slope    brier      ece
         Overall              -0.030786           0.081434 0.224863 0.080370
African-American              -0.045822           0.313613 0.219964 0.066759
           Asian     

In [15]:
mitigated_w = audit.mitigate(method=WassersteinPostprocessing(random_state=SEED), calibration_data=calib_data)
d_after = mitigated_w[a0].discrimination()
d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
    ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]]

,gap_metric,gap_value,gap_ci_low,gap_ci_high
6,equal_opportunity_difference,0.600000,NaN,NaN
7,predictive_equality,0.811659,NaN,NaN
8,equalized_odds_gap,0.811659,NaN,NaN
9,statistical_parity,0.660714,NaN,NaN


## 12. HTML report

In [16]:
REPORT = HERE / "reports" / f"{NAME}.html"
REPORT.parent.mkdir(exist_ok=True)
audit.report(out=REPORT, title=f"{NAME}: isitfair audit",
             metadata={"Dataset": NAME, "Model": "LogisticRegression",
                       "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
             mitigation_audits=mitigated_gr)
print("wrote", REPORT.relative_to(HERE.parent))

wrote extra_examples\reports\compas.html
